# Generate EPI dataset

In [2]:
import os
from PIL import Image
import numpy as np
import imageio
import numpy as np
import matplotlib.pyplot as plt
import sys
from skimage import io

### Wind5-LID dataset

The format input of this dataset is .bmp files

The structure for the input dataset  is described bellow:
```
Win5-LID/
├── Compressed/               
│   └── Real/         
│       └── EPICNN_Bikes.bmp
│       └── EPICNN_Flowers.bmp
│       └── HEVC_Vespa_34.bmp
│   └── Synthetic/        
│       └── EPICNN_dishes.bmp
│       └── HEVC_greek_29.bmp
│       └── HEVC_rosemary_24.bmp
```

This script reads the folder structure and generates the output directories. A class called win5lid.py handles the management of data, preparing it for both training and testing processes.

In [ ]:
root='Win5-LID/Compressed'
folders = os.listdir(root)

for type in folders:
    img_dir = os.path.join(root, type)
    if os.path.isdir(img_dir):
        dists = os.listdir(img_dir)
        for img in dists:
            im = imageio.imread(f"{img_dir}/{img}")
            if(type == 'Synthetic'):
                data = np.transpose(np.reshape(im,[512, 9, 512,  9, 3]),(1, 3, 0, 2, 4))
            else:
                data = np.transpose(np.reshape(im,[434, 9, 625, 9, 3]),(1, 3, 0, 2, 4))

            h, w, c = data[0, 0, :, :, :].shape
            # Create the output directory structure
            os.makedirs(
                f'win5lidHor/{img.split(".")[0].split("_")[1]}/', exist_ok=True
            )  # Create the necessary directories
            os.makedirs(
                f'win5lidVert/{img.split(".")[0].split("_")[1]}/', exist_ok=True
            )  # Create the necessary directories


            #horizontal
            list_img =[]
            testeImg=[]
            for i in range(0, 9):  
                list_img.append(np.array(data[i, 4, :, :, :]))

            for l in range (0,h):
                for j in range(0,9):
                    testeImg.append(list_img[j][l,:,:])
            io.imsave(f'win5lidHor/{img.split(".")[0].split("_")[1]}/{img.split(".")[0]}.png',np.reshape(testeImg,[9*h, w, 3] ))


            #vertical    
            list_img =[]
            testeImg=[]
            for i in range(0, 9):  
                list_img.append(np.array(data[4,i, :, :, :]))

            for l in range (0,w):
                for j in range(0,9):
                    testeImg.append(list_img[j][:,l,:])

            io.imsave(f'win5lidVert/{img.split(".")[0].split("_")[1]}/{img.split(".")[0]}.png',np.reshape(testeImg,[9*w, h, 3]))
    


### VALID Dataset

The structure for the input dataset is based on valid 10bits. The input layout exemplification is following as:
```
VALID/Compressed/
├── HEVC/               
│   └── I01/         
│       └── I01P1R1/
│       └── I01P1R2/
│   └── I10/        
│       └── I10P1R1/
│       └── I10P1R2/
├── P3/               
│   └── I01/         
│       └── I01P3R1/
│       └── I01P3R2/
│   └── I10/        
│       └── I10P3R1/
│       └── I10P3R2/
```

This script reads the folder structure and generates the output directories. A class called valid.py handles the management of data, preparing it for both training and testing processes.

In [ ]:
base_input_dir='VALID/Compressed'

imgs = os.listdir(base_input_dir)
for img in imgs:
    img_dir = os.path.join(base_input_dir, img)
    if os.path.isdir(img_dir):
        dists = os.listdir(img_dir)
        for dist in dists:
            types = os.listdir(os.path.join(img_dir, dist))
            # Create the output directory structure
            os.makedirs(
                f'VALIDHor/{dist}/', exist_ok=True
            )  # Create the necessary directories
            os.makedirs(
                f'VALIDVert/{dist}/', exist_ok=True
            )  # Create the necessary directories

            for type in types:
                #horizontal
                list_img =[]
                testeImg=[]
                
                for i in range (1, 14):
                    j = 12
                    image_path = os.path.join(
                                            img_dir,
                                            dist,
                                            type,
                                            f"{i:03d}_{j:03d}.ppm",
                                        )
                    image = Image.open(image_path)
                    
                    list_img.append(np.array(image))    
                for l in range (0,434):
                    for j in range(0,13):
                        testeImg.append(list_img[j][l,:,:])
                io.imsave(f'VALIDHor/{dist}/{type}.png',np.reshape(testeImg,[13*434, 626, 3] ))

                #vertical    
                list_img =[]
                testeImg=[]
                for j in range (1, 14):
                    i = 12
                    image_path = os.path.join(
                                                img_dir,
                                                dist,
                                                type,
                                                f"{i:03d}_{j:03d}.ppm",
                                            )
                    image = Image.open(image_path)
                        
                    list_img.append(np.array(image))    

                for l in range (0,626):
                    for j in range(0,13):
                        testeImg.append(list_img[j][:,l,:])

                io.imsave(f'VALIDVert/{dist}/{type}.png',np.reshape(testeImg,[13*626, 434, 3]))
            

### LFDD Dataset

The structure for the input dataset is considerate as the original structure. Below is an example illustrating the directory layout:
```
lfdd/
├── boxes/boxes/              
│   └── boxes_av1_01/  
│   └── boxes_av1_05/         
│   └── boxes_bpg_625/         
├── cotton/cotton/              
│   └── cotton_av1_01/  
│   └── cotton_av1_05/         
│   └── cotton_bpg_625/  
├── kitchen/kitchen/              
│   └── kitchen_av1_01/  
│   └── kitchen_av1_05/         
│   └── kitchen_bpg_625/  
```

This script reads the folder structure and generates the output directories. The data is processed and stored within the main data directory. A class called lfdd.py handles the management of data, preparing it for both training and testing processes.

In [ ]:
base_input_dir="lfdd"
imgs = os.listdir(base_input_dir)
hor_indx = {5, 14, 23, 32, 41, 50, 59, 68, 77}
vert_indx = {7, 38, 39, 40, 41, 42, 43, 44, 45}

for img in imgs:
    img_dir = os.path.join(base_input_dir, img, img)
    if os.path.isdir(img_dir):
        dists = os.listdir(img_dir)
        for dist in dists:
            if os.path.isdir(os.path.join(img_dir, dist)):
                types = os.listdir(os.path.join(img_dir, dist))
                # Create the output directory structure
                os.makedirs(
                    f'LFDDHor/{img}/', exist_ok=True
                )  # Create the necessary directories
                os.makedirs(
                    f'LFDDVert/{img}', exist_ok=True
                )  # Create the necessary directories

                #horizontal
                list_img =[]
                testeImg=[]
                
                for i in hor_indx:
                    image_path = os.path.join(
                                    img_dir, dist, f"input_Cam0{i:02d}.png"
                                )
                    image = Image.open(image_path)
                    
                    list_img.append(np.array(image))    
                for l in range (0,image.size[1]):
                    for j in range(0,9):
                        testeImg.append(list_img[j][l,:,:])
                io.imsave(f'LFDDHor/{img}/{dist}.png',np.reshape(testeImg,[9*image.size[1], image.size[0], 3] ))

                #vertical  
                list_img =[]
                testeImg=[]
                
                for i in vert_indx:
                    image_path = os.path.join(
                                    img_dir, dist, f"input_Cam0{i:02d}.png"
                                )
                    image = Image.open(image_path)
                    
                    list_img.append(np.array(image))    
                for l in range (0,image.size[0]):
                    for j in range(0,9):
                        testeImg.append(list_img[j][:,l,:])
                io.imsave(f'LFDDVert/{img}/{dist}.png',np.reshape(testeImg,[9*image.size[0], image.size[1], 3]))


### Robust Large-scale Dataset

The structure for the input dataset is considerate as the original structure. Below is an example illustrating the directory layout:
```
Light Field Image/
├── Bee_2__Decoded/               
│   └── Gaussian Blur/         
│       └── 1_5_0_0/
│       └── 5_5_0_0/
│   └── HEVC/        
│       └── qp22/
│       └── qp32/
├── bicycle/               
│   └── Gaussian Blur/         
│       └── 1_5_0_0/
│       └── 5_5_0_0/
│   └── HEVC/        
│       └── qp22/
│       └── qp32/
```

This script reads the folder structure and generates the output directories, maintaining the same hierarchical organization. The data is processed and stored within the main data directory. A class called RobustLarge.py handles the management of data, preparing it for both training and testing processes.

In [ ]:
base_input_dir='Light Field Image'
hor_indx = {5, 14, 23, 32, 41, 50, 59, 68, 77}
vert_indx = {7, 38, 39, 40, 41, 42, 43, 44, 45}

imgs = os.listdir(base_input_dir)
for img in imgs:
    img_dir = os.path.join(base_input_dir, img)        
    if os.path.isdir(img_dir):
        dists = os.listdir(img_dir)
        for dist in dists:
            print(img_dir, dist)
            if 'Reference' not in dist:
                types = os.listdir(os.path.join(img_dir, dist))
                for type in types:
                    # Create the output directory structure
                    os.makedirs(f'RobustHor/{img}', exist_ok=True)  # Create the necessary directories
                    #Horizontal
                    list_img =[]
                    testeImg=[]
                    
                    for i in hor_indx:
                        try:
                            imaux=Image.open(f"{img_dir}/{dist}/{type}/{i}.png")
                        except:
                            imaux=Image.open(f"{img_dir}/{dist}/{type}/{i:02d}.png")
                        list_img.append(np.array(imaux))    
                    for l in range (0,imaux.size[1]):
                        for j in range(0,9):
                            testeImg.append(list_img[j][l,:,:])
                    io.imsave(f'RobustHor/{img}/{dist}_{type}.png',np.reshape(testeImg,[9*imaux.size[1], imaux.size[0], 3] ))


                    #Vertical  
                    os.makedirs(f'RobustVert/{img}', exist_ok=True)  # Create the necessary directories  
                    list_img =[]
                    testeImg=[]
                    for j in vert_indx:
                        try:
                            imaux=Image.open(f"{img_dir}/{dist}/{type}/{j}.png")
                        except:
                            imaux=Image.open(f"{img_dir}/{dist}/{type}/{j:02d}.png")
                        list_img.append(np.array(imaux))    

                    for l in range (0,imaux.size[0]):
                        for j in range(0,9):
                            testeImg.append(list_img[j][:,l,:])

                    io.imsave(f'RobustVert/{img}/{dist}_{type}.png',np.reshape(testeImg,[9*imaux.size[0], imaux.size[1], 3]))